# Training runs on Kaggle

Thin wrapper over the repo: clones it, installs the `ultralytics` and `rfdetr` versions from its `uv.lock` (Kaggle's own torch stays), and for every entry in `RUNS` runs `scripts/train.py` and then `scripts/predict.py` on test A.

Settings: accelerator GPU T4 x2 (training uses one), internet on, dataset `wojtekkosmalski/d2s-yolo` attached. Run in the background with Save Version → Save & Run All. Push the code to GitHub first: the notebook clones `master`.

Output in `/kaggle/working/runs/`: one folder per run with its weights and `pred_test_a*.json`, and `runs.csv` with one row per session (versions, GPU, epochs, time, peak GPU memory, last validation mAP). A 12 h session fits one RF-DETR run (about 9 h) or a few YOLO runs (about 1.5-2 h each).

To continue runs cut off by the 12 h limit: attach this notebook's previous version output as input, set `RESUME_FROM` to its `runs/` folder and run the same `RUNS` again.

In [ ]:
DATA = "/kaggle/input/datasets/wojtekkosmalski/d2s-yolo"
RESUME_FROM = None  # e.g. the runs/ folder of a previous version attached as input
RUNS = [  # R1/c60, seed 0; RF-DETR needs a session of its own
    "--model yolo11s --regime R1 --labels c60 --seed 0 --epochs 100",
    "--model yolo26s --regime R1 --labels c60 --seed 0 --epochs 100",
    # "--model rfdetr-s --regime R1 --labels c60 --seed 0 --epochs 100",
]

In [ ]:
import tomllib

!git clone --depth 1 https://github.com/chewmanji/detr-vs-yolo-occlusion.git repo
!git -C repo log -1 --format='commit %H %s'
with open("repo/uv.lock", "rb") as f:
    pins = {p["name"]: p["version"] for p in tomllib.load(f)["package"]}
ul, rf = pins["ultralytics"], pins["rfdetr"]
!pip install -q ultralytics=={ul} "rfdetr[train]=={rf}"

In [ ]:
import shutil

import torch

assert torch.cuda.is_available(), "turn on the GPU accelerator"
print(torch.__version__, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
if RESUME_FROM:
    shutil.copytree(RESUME_FROM, "runs", dirs_exist_ok=True)

In [ ]:
import csv

failed = []
for run in RUNS:  # a failed run does not stop the others
    !python repo/scripts/train.py {run} --data {DATA} --out runs
    if _exit_code:  # noqa: F821, set by IPython after each ! command
        failed.append(run)
        continue
    with open("runs/runs.csv") as f:
        name = list(csv.DictReader(f))[-1]["run"]  # train.py appends its row when it finishes
    !python repo/scripts/predict.py runs/{name} --data {DATA}
    if _exit_code:  # noqa: F821
        failed.append(f"predict {name}")
print("failed:", failed)
!cat runs/runs.csv